In [ ]:
import csv
import json
import random

data = []
with open("大众点评数据/train (2).csv", 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for idx, row in enumerate(reader):
        data.append({
            "idx": idx,
            "context": row["sentence"],
            "label": int(row["label"])
        })

random.shuffle(data) #打乱数据顺序
n = len(data)
train_data = data[:int(0.8*n)]
val_data = data[int(0.8*n):int(0.9*n)]
test_data = data[int(0.9*n):]

with open("大众点评数据/train.json", 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
with open("大众点评数据/val.json", 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)
with open("大众点评数据/test.json", 'w', encoding='utf-8') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)



In [ ]:
from functools import partial
data_output_path = ["大众点评数据/train.json","大众点评数据/val.json","大众点评数据/test.json"]
from modelscope import GPT2Tokenizer, GPT2LMHeadModel
from datasets import load_dataset
hf_model_path = 'Fengshenbang/Wenzhong-GPT2-110M-chinese-v2'
tokenizer = GPT2Tokenizer.from_pretrained(hf_model_path)
# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch):
    return tokenizer(batch['context'])

dataset = load_dataset('json', data_files={
    'train': data_output_path[0],
    'valid': data_output_path[1],
    'test': data_output_path[2]
})
# 查看结构
print(dataset['train'][0])
# 输出：{'idx': 19087, 'context': '点了榛子可可华夫 9元 比想象的好吃 听介绍里面是慕斯和果酱 但是冷藏过 冰冰的 感觉像冰淇淋 不是很甜 很好吃 就是太小了 两三口就没啦', 'label': 0}

map_kwargs = {
    'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
    'batch_size': 512,  # 每批 512 个样本
    'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
}

# 对训练集和验证集应用 map
tokenized_dataset_train = dataset["train"].map(tokenize, **map_kwargs)
tokenized_dataset_val = dataset["valid"].map(tokenize, **map_kwargs)

print(tokenized_dataset_train[0])
print(tokenized_dataset_val[0])


In [ ]:

for i, seq in enumerate(tokenized_dataset_train[5:10]['input_ids']):
    print(f'{i + 1}: {tokenizer.decode(seq)}')


#清理数据长度国小的
def filter_short_samples(batch, min_length=10):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)


# 对训练集和验证集应用过滤函数
filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

#放入torch
import torch

filtered_dataset_train.set_format(type='torch')
filtered_dataset_val.set_format(type='torch')
print(filtered_dataset_train[0])
print(filtered_dataset_val[0])
# 计算验证集中 input_ids 的最大长度
max_len = max(len(seq) for seq in filtered_dataset_val['input_ids'])
print(f"验证集最大序列长度: {max_len}")

In [ ]:
#填充
print(tokenizer.pad_token)
print(tokenizer.eos_token)
#将填充标记设置为 EOS 标记
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling
# mlm=False，将数据整理成“因果语言建模”需要的数据格式
# “因果语言建模”就是“预测下一个token”类型的任务，也就是gpt风格的自回归模型
# 如果mlm=True，那么数据整理成bert风格的任务所需的数据格式
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False) # labels

dataloader_params = {
    'batch_size': 16, #根据需要调整批量大小
    'collate_fn': data_collator
}

train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

print(len(train_dataloader))

batch = next(iter(train_dataloader))
print(batch.keys())
print(batch['input_ids'].shape)
print(batch['input_ids'][0])
print(batch['labels'][0])
print(batch['attention_mask'][0])

In [ ]:
#加载模型监督微调
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GPT2LMHeadModel.from_pretrained(hf_model_path).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
#训练一个epoch
num_epochs = 1

In [ ]:
# 训练循环
#先评估一下初始模型在验证集上的表现
def validate(epoch):
    model.eval()
    total_loss = 0.0
    for i, batch in enumerate(val_dataloader):
        batch = batch.to(device)
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs.loss # 损失
            total_loss += loss.item()
    print(f'val_loss at {epoch} epoch:', total_loss / len(val_dataloader))
validate(1)
#正式训练
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for i, batch in enumerate(train_dataloader):
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss # 损失
        total_loss += loss.item()
        optimizer.zero_grad() # 梯度清零
        loss.backward() # 反向传播
        optimizer.step() # 更新参数
        print(f'train_loss at {i} epoch:', loss.item())
    print("train_loss at {epoch} epoch:", total_loss / len(train_dataloader))
    validate(epoch+1)

In [ ]:
#保存微调后的模型
model.save_pretrained('models/fine_tuned_gpt2_大众')
tokenizer.save_pretrained('models/fine_tuned_gpt2_大众')